# Cross-Validation with Missing Events: Visual Demonstration

This notebook demonstrates how the two zero-event strategies handle missing stimulus events.

**Setup:**
- 1 voxel (for clarity)
- 2 runs, 300 TRs each, TR=1s
- 4 events, each occurs 2x per run (well-separated)
- Events are 1s duration, convolved with canonical HRF
- Uses actual fastfuncsim CV code

**Key Comparison:**
- **"zero" strategy**: Uses zero beta for missing events → can't predict that variance
- **"nuisance" strategy**: Projects out unpredictable events from test → predicts what it can

**Scenarios:**
1. Baseline: All events present
2. Train missing Event 2: Can't learn β₂
3. Test missing Event 2: Learn β₂ but Event 2 not in test
4. Both missing different: Event 1 missing from test, Event 2 from train

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import torch
from scipy.stats import gamma
from typing import Tuple, List

# Import actual CV code
from fastfuncsim.xval import compute_xval_r2, generate_cv_splits
from fastfuncsim.glm_core import fit_glm_ols

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("Loaded fastfuncsim CV code successfully!")

## Helper Functions for Data Generation

In [ ]:
def canonical_hrf(tr: float = 1.0, duration: float = 32.0) -> np.ndarray:
    """Create canonical HRF (SPM-style double gamma)."""
    t = np.arange(0, duration, tr)
    
    # Positive gamma
    peak1, scale1 = 6.0, 1.0
    pos = gamma.pdf(t, peak1, scale=scale1)
    
    # Negative gamma (undershoot)
    peak2, scale2 = 16.0, 1.0
    neg = gamma.pdf(t, peak2, scale=scale2)
    
    # Combine
    hrf = pos - neg / 6.0
    hrf = hrf / hrf.max()  # Normalize
    
    return hrf


def create_event_design(
    n_timepoints: int,
    event_onsets: List[int],
    tr: float = 1.0,
    duration: float = 1.0,
) -> np.ndarray:
    """Create HRF-convolved design for a single event type."""
    # Stick function
    stick = np.zeros(n_timepoints)
    for onset in event_onsets:
        if 0 <= onset < n_timepoints:
            stick[onset] = duration / tr
    
    # Convolve with HRF
    hrf = canonical_hrf(tr=tr, duration=32.0)
    convolved = np.convolve(stick, hrf, mode='full')[:n_timepoints]
    
    return convolved


def create_full_design(
    n_timepoints: int,
    n_events: int,
    events_per_run: int = 2,
    missing_events: List[int] = None,
    seed: int = 42,
) -> Tuple[np.ndarray, List[List[int]]]:
    """
    Create design matrix with well-separated events.
    
    Returns
    -------
    design : np.ndarray
        Design matrix (n_timepoints, n_events)
    onset_times : List[List[int]]
        Onset times for each event
    """
    if missing_events is None:
        missing_events = []
    
    rng = np.random.RandomState(seed)
    design = np.zeros((n_timepoints, n_events))
    onset_times = []
    
    # Total events across all types
    total_events = n_events * events_per_run
    
    # Create well-separated time bins
    # Leave buffer at start (30 TRs) and end (30 TRs)
    usable_time = n_timepoints - 60
    bin_size = usable_time // total_events
    
    # Assign bins to events
    all_bins = list(range(total_events))
    rng.shuffle(all_bins)
    
    bin_idx = 0
    for event_idx in range(n_events):
        if event_idx in missing_events:
            onset_times.append([])
        else:
            onsets = []
            for _ in range(events_per_run):
                # Get bin
                bin_num = all_bins[bin_idx]
                bin_idx += 1
                
                # Random position within bin
                bin_start = 30 + bin_num * bin_size
                onset = bin_start + rng.randint(0, max(1, bin_size - 10))
                onsets.append(onset)
            
            onset_times.append(sorted(onsets))
            design[:, event_idx] = create_event_design(n_timepoints, onsets)
    
    return design, onset_times


def generate_fmri_data(
    design: np.ndarray,
    true_betas: np.ndarray,
    noise_std: float = 0.5,
    seed: int = 42,
) -> np.ndarray:
    """Generate synthetic fMRI data."""
    rng = np.random.RandomState(seed)
    signal = design @ true_betas
    noise = rng.randn(len(signal)) * noise_std
    return signal + noise


print("Helper functions defined!")

## Setup: Common Parameters

In [ ]:
# Parameters
n_timepoints = 300  # TRs per run (5 minutes)
n_events = 4  # Event types
events_per_run = 2  # Each event occurs 2x per run
tr = 1.0  # seconds
noise_std = 0.5  # Noise level

# True betas (same for both runs!)
true_betas = np.array([2.0, 1.5, 1.0, 0.5])  # Event 0 strongest, Event 3 weakest

# Colors and names
event_colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
event_names = ['Event 0', 'Event 1', 'Event 2', 'Event 3']

print(f"True betas: {true_betas}")
print(f"Events per run: {events_per_run}")
print(f"Total TRs per run: {n_timepoints}")
print(f"Total duration: {n_timepoints * tr / 60:.1f} minutes per run")

## Helper: Run CV with Both Strategies

In [ ]:
def run_cv_scenario(
    train_missing: List[int],
    test_missing: List[int],
    verbose: bool = True,
) -> Tuple[dict, dict, dict]:
    """
    Run CV with both strategies for a given missing event scenario.
    
    Returns
    -------
    data_dict : dict
        Contains train/test data and designs
    results_zero : dict
        Results from "zero" strategy
    results_nuisance : dict
        Results from "nuisance" strategy
    """
    # Create designs
    design_train, onsets_train = create_full_design(
        n_timepoints, n_events, events_per_run, train_missing, seed=42
    )
    design_test, onsets_test = create_full_design(
        n_timepoints, n_events, events_per_run, test_missing, seed=43
    )
    
    # Generate data
    data_train = generate_fmri_data(design_train, true_betas, noise_std, seed=100)
    data_test = generate_fmri_data(design_test, true_betas, noise_std, seed=101)
    
    # Package data (1 voxel, n_timepoints)
    data_train_2d = data_train.reshape(1, -1)
    data_test_2d = data_test.reshape(1, -1)
    data_concat = np.concatenate([data_train_2d, data_test_2d], axis=1)
    
    # Full design matrix (train + test)
    design_full = np.vstack([design_train, design_test])
    
    # Convert to torch
    data_torch = torch.from_numpy(data_concat).float()
    design_torch = torch.from_numpy(design_full).float()
    
    # Run info
    run_starts = [0, n_timepoints]
    stim_indices = list(range(n_events))
    nuisance_indices = []  # No nuisance for simplicity
    
    # CV splits: train on run 0, test on run 1
    cv_splits = [([0], [1])]
    
    # Run with "zero" strategy
    results_zero = compute_xval_r2(
        data=data_torch,
        design_matrix=design_torch,
        run_starts=run_starts,
        stim_indices=stim_indices,
        nuisance_indices=nuisance_indices,
        cv_splits=cv_splits,
        metric='cod',
        zero_event_strategy='zero',
        device=torch.device('cpu'),
        verbose=False,
    )
    
    # Run with "nuisance" strategy
    results_nuisance = compute_xval_r2(
        data=data_torch,
        design_matrix=design_torch,
        run_starts=run_starts,
        stim_indices=stim_indices,
        nuisance_indices=nuisance_indices,
        cv_splits=cv_splits,
        metric='cod',
        zero_event_strategy='nuisance',
        device=torch.device('cpu'),
        verbose=False,
    )
    
    # Also fit betas manually for visualization
    train_present = ~np.all(design_train == 0, axis=0)
    design_train_fit = design_train[:, train_present]
    
    if design_train_fit.shape[1] > 0:
        betas_fit_vals = np.linalg.lstsq(design_train_fit, data_train, rcond=None)[0]
        betas_fit = np.zeros(n_events)
        betas_fit[train_present] = betas_fit_vals
    else:
        betas_fit = np.zeros(n_events)
    
    # Predictions
    pred_zero = design_test @ betas_fit
    
    # For nuisance strategy: project out unpredictable events
    test_present = ~np.all(design_test == 0, axis=0)
    predictable = train_present & test_present
    unpredictable = train_present & ~test_present  # Events in train but not test
    test_only = ~train_present & test_present      # Events in test but not train
    
    # Events to project out: both unpredictable AND test-only
    events_to_project = unpredictable | test_only
    
    if np.any(events_to_project):
        # Project out both types of unpredictable variance from test data
        test_to_project = design_test[:, events_to_project]
        if test_to_project.shape[1] > 0 and np.any(test_to_project != 0):
            P = test_to_project @ np.linalg.lstsq(test_to_project, np.eye(len(test_to_project)), rcond=None)[0]
            data_test_proj = data_test - P @ data_test
        else:
            data_test_proj = data_test
        
        # Predict with only predictable betas
        pred_nuisance = design_test[:, predictable] @ betas_fit[predictable]
    else:
        data_test_proj = data_test
        pred_nuisance = pred_zero
    
    # Compute R² manually
    def compute_r2(y_true, y_pred):
        ss_res = np.sum((y_true - y_pred) ** 2)
        ss_tot = np.sum((y_true - y_true.mean()) ** 2)
        return 1 - (ss_res / ss_tot)
    
    r2_zero = compute_r2(data_test, pred_zero)
    r2_nuisance = compute_r2(data_test_proj, pred_nuisance)
    
    if verbose:
        print(f"Train missing: {train_missing}")
        print(f"Test missing:  {test_missing}")
        print(f"True betas:    {true_betas}")
        print(f"Fitted betas:  {betas_fit}")
        print(f"")
        print(f"R² (zero):     {r2_zero:.4f}")
        print(f"R² (nuisance): {r2_nuisance:.4f}")
        print(f"Improvement:   {r2_nuisance - r2_zero:.4f}")
    
    data_dict = {
        'data_train': data_train,
        'data_test': data_test,
        'data_test_proj': data_test_proj,
        'design_train': design_train,
        'design_test': design_test,
        'onsets_train': onsets_train,
        'onsets_test': onsets_test,
        'betas_fit': betas_fit,
        'pred_zero': pred_zero,
        'pred_nuisance': pred_nuisance,
        'r2_zero': r2_zero,
        'r2_nuisance': r2_nuisance,
    }
    
    return data_dict, results_zero, results_nuisance


print("CV runner defined!")

## Plotting Function

In [ ]:
def plot_scenario(data_dict, title, train_missing, test_missing):
    """
    Plot train/test data and both prediction strategies.
    """
    fig, axes = plt.subplots(4, 1, figsize=(16, 14))
    time = np.arange(n_timepoints) * tr
    
    data_train = data_dict['data_train']
    data_test = data_dict['data_test']
    data_test_proj = data_dict['data_test_proj']
    design_train = data_dict['design_train']
    design_test = data_dict['design_test']
    onsets_train = data_dict['onsets_train']
    onsets_test = data_dict['onsets_test']
    betas_fit = data_dict['betas_fit']
    pred_zero = data_dict['pred_zero']
    pred_nuisance = data_dict['pred_nuisance']
    r2_zero = data_dict['r2_zero']
    r2_nuisance = data_dict['r2_nuisance']
    
    # Panel 1: Train data
    ax = axes[0]
    ax.plot(time, data_train, 'k-', linewidth=2, label='Train Data', alpha=0.7)
    ax.plot(time, design_train @ true_betas, 'k--', linewidth=1, label='True Signal', alpha=0.5)
    
    for event_idx in range(n_events):
        for onset in onsets_train[event_idx]:
            ax.axvline(onset * tr, color=event_colors[event_idx], alpha=0.4, linestyle='-', linewidth=2)
    
    if train_missing:
        missing_str = ', '.join([f'Event {i}' for i in train_missing])
        ax.text(0.02, 0.95, f'⚠️ MISSING: {missing_str}', transform=ax.transAxes,
                fontsize=12, fontweight='bold', color='red',
                verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))
    
    ax.set_xlabel('Time (s)', fontsize=12)
    ax.set_ylabel('Signal', fontsize=12)
    ax.set_title('Train Run: Fit betas on this data', fontsize=14, fontweight='bold')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    
    # Panel 2: Test data
    ax = axes[1]
    ax.plot(time, data_test, 'k-', linewidth=2, label='Test Data', alpha=0.7)
    ax.plot(time, design_test @ true_betas, 'k--', linewidth=1, label='True Signal', alpha=0.5)
    
    for event_idx in range(n_events):
        for onset in onsets_test[event_idx]:
            ax.axvline(onset * tr, color=event_colors[event_idx], alpha=0.4, linestyle='-', linewidth=2)
    
    if test_missing:
        missing_str = ', '.join([f'Event {i}' for i in test_missing])
        ax.text(0.02, 0.95, f'⚠️ MISSING: {missing_str}', transform=ax.transAxes,
                fontsize=12, fontweight='bold', color='red',
                verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))
    
    ax.set_xlabel('Time (s)', fontsize=12)
    ax.set_ylabel('Signal', fontsize=12)
    ax.set_title('Test Run: Predict this data', fontsize=14, fontweight='bold')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    
    # Panel 3: "Zero" strategy prediction
    ax = axes[2]
    ax.plot(time, data_test, 'k-', linewidth=2, label='Test Data', alpha=0.5)
    ax.plot(time, pred_zero, 'r-', linewidth=2, label='Prediction (zero)', alpha=0.8)
    ax.plot(time, data_test - pred_zero, 'gray', linewidth=1, label='Residual', alpha=0.5)
    
    # Highlight missing events if they cause problems
    if train_missing:
        for event_idx in train_missing:
            for onset in onsets_test[event_idx]:
                ax.axvspan(onset*tr, (onset+15)*tr, color='red', alpha=0.1)
    
    ax.text(0.98, 0.95, f'Strategy: "zero"\nR² = {r2_zero:.4f}', 
            transform=ax.transAxes, fontsize=11, fontweight='bold',
            verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.7))
    
    ax.set_xlabel('Time (s)', fontsize=12)
    ax.set_ylabel('Signal', fontsize=12)
    ax.set_title('Strategy 1: "zero" - Use zero beta for missing events', fontsize=14, fontweight='bold')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
    
    # Panel 4: "Nuisance" strategy prediction
    ax = axes[3]
    
    # If we projected, show projected data
    if not np.allclose(data_test, data_test_proj):
        ax.plot(time, data_test, 'k-', linewidth=1, label='Test Data (original)', alpha=0.3)
        ax.plot(time, data_test_proj, 'k-', linewidth=2, label='Test Data (projected)', alpha=0.7)
        ax.plot(time, pred_nuisance, 'b-', linewidth=2, label='Prediction (nuisance)', alpha=0.8)
        ax.plot(time, data_test_proj - pred_nuisance, 'gray', linewidth=1, label='Residual', alpha=0.5)
    else:
        ax.plot(time, data_test, 'k-', linewidth=2, label='Test Data', alpha=0.5)
        ax.plot(time, pred_nuisance, 'b-', linewidth=2, label='Prediction (nuisance)', alpha=0.8)
        ax.plot(time, data_test - pred_nuisance, 'gray', linewidth=1, label='Residual', alpha=0.5)
    
    improvement = r2_nuisance - r2_zero
    color = 'lightgreen' if improvement > 0.01 else 'lightyellow'
    
    ax.text(0.98, 0.95, f'Strategy: "nuisance"\nR² = {r2_nuisance:.4f}\nΔR² = {improvement:+.4f}', 
            transform=ax.transAxes, fontsize=11, fontweight='bold',
            verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor=color, alpha=0.7))
    
    ax.set_xlabel('Time (s)', fontsize=12)
    ax.set_ylabel('Signal', fontsize=12)
    ax.set_title('Strategy 2: "nuisance" - Project out unpredictable events, predict the rest', 
                 fontsize=14, fontweight='bold')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
    
    # Overall title and legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=event_colors[i], alpha=0.5, 
                             label=f'{event_names[i]} (β={true_betas[i]:.1f}, fit={betas_fit[i]:.2f})') 
                       for i in range(n_events)]
    fig.legend(handles=legend_elements, loc='upper center', ncol=4, fontsize=10, 
               bbox_to_anchor=(0.5, 0.99))
    
    fig.suptitle(title, fontsize=16, fontweight='bold', y=0.995)
    
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()


print("Plotting function defined!")

## Scenario 1: Baseline - All Events Present

In [ ]:
print("\n" + "="*70)
print("SCENARIO 1: Baseline - All events present")
print("="*70)

data_dict, res_zero, res_nuisance = run_cv_scenario(
    train_missing=[],
    test_missing=[],
    verbose=True,
)

plot_scenario(
    data_dict,
    'Scenario 1: Baseline - All Events Present',
    train_missing=[],
    test_missing=[],
)

## Scenario 2: Train Missing Event 2

**Key issue**: Can't learn β₂, so can't predict Event 2 signal in test.

**Expected**: Both strategies should be similar (can't predict what we didn't learn).

In [ ]:
print("\n" + "="*70)
print("SCENARIO 2: Train missing Event 2")
print("="*70)

data_dict, res_zero, res_nuisance = run_cv_scenario(
    train_missing=[2],
    test_missing=[],
    verbose=True,
)

plot_scenario(
    data_dict,
    'Scenario 2: Train Missing Event 2 (cannot learn β₂)',
    train_missing=[2],
    test_missing=[],
)

## Scenario 3: Test Missing Event 2

**Key issue**: Learn β₂ from train, but Event 2 not in test data.

**Expected**: 
- **"zero" strategy**: Uses β₂ anyway (wrong!), adds unexplained variance
- **"nuisance" strategy**: Projects out Event 2 from test before prediction → BETTER!

In [ ]:
print("\n" + "="*70)
print("SCENARIO 3: Test missing Event 2")
print("="*70)

data_dict, res_zero, res_nuisance = run_cv_scenario(
    train_missing=[],
    test_missing=[2],
    verbose=True,
)

plot_scenario(
    data_dict,
    'Scenario 3: Test Missing Event 2 (learned β₂ but Event 2 not in test)',
    train_missing=[],
    test_missing=[2],
)

## Scenario 4: Both Missing Different Events

**Complex case**:
- Event 1 missing from test (learned β₁ but shouldn't use)
- Event 2 missing from train (can't learn β₂)

**Expected**:
- **"zero" strategy**: Can't predict Event 2, wrongly predicts Event 1
- **"nuisance" strategy**: Projects out Event 1, doesn't predict Event 2 → BETTER!

In [ ]:
print("\n" + "="*70)
print("SCENARIO 4: Both missing different events")
print("="*70)

data_dict, res_zero, res_nuisance = run_cv_scenario(
    train_missing=[2],
    test_missing=[1],
    verbose=True,
)

plot_scenario(
    data_dict,
    'Scenario 4: Event 2 missing from train, Event 1 missing from test',
    train_missing=[2],
    test_missing=[1],
)

## Summary Comparison

In [ ]:
# Run all scenarios
scenarios = [
    ('Baseline\n(all present)', [], []),
    ('Train Missing\nEvent 2', [2], []),
    ('Test Missing\nEvent 2', [], [2]),
    ('Both Missing\nDifferent', [2], [1]),
]

r2_zero_all = []
r2_nuisance_all = []

for name, train_missing, test_missing in scenarios:
    data_dict, _, _ = run_cv_scenario(train_missing, test_missing, verbose=False)
    r2_zero_all.append(data_dict['r2_zero'])
    r2_nuisance_all.append(data_dict['r2_nuisance'])

# Plot comparison
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(scenarios))
width = 0.35

bars1 = ax.bar(x - width/2, r2_zero_all, width, label='"zero" strategy', 
               color='lightcoral', alpha=0.8, edgecolor='black')
bars2 = ax.bar(x + width/2, r2_nuisance_all, width, label='"nuisance" strategy', 
               color='lightgreen', alpha=0.8, edgecolor='black')

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{height:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Add improvement annotations
for i, (r2_z, r2_n) in enumerate(zip(r2_zero_all, r2_nuisance_all)):
    improvement = r2_n - r2_z
    if abs(improvement) > 0.005:
        y_pos = max(r2_z, r2_n) + 0.05
        color = 'green' if improvement > 0 else 'red'
        ax.text(i, y_pos, f'Δ={improvement:+.3f}', ha='center', va='bottom',
                fontsize=10, fontweight='bold', color=color)

ax.set_xlabel('Scenario', fontsize=14, fontweight='bold')
ax.set_ylabel('R² (out-of-sample)', fontsize=14, fontweight='bold')
ax.set_title('Cross-Validation R²: "zero" vs "nuisance" Strategies', fontsize=16, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([s[0] for s in scenarios], fontsize=11)
ax.set_ylim([0, max(max(r2_zero_all), max(r2_nuisance_all)) * 1.15])
ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
ax.legend(fontsize=12, loc='upper right')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Print summary
print("\n" + "="*70)
print("SUMMARY: R² Comparison Across Strategies")
print("="*70)
for (name, _, _), r2_z, r2_n in zip(scenarios, r2_zero_all, r2_nuisance_all):
    improvement = r2_n - r2_z
    print(f"{name.replace(chr(10), ' '):30s} zero={r2_z:.4f}  nuisance={r2_n:.4f}  Δ={improvement:+.4f}")

print("\nKey Insight:")
print("  - 'nuisance' strategy helps when test is missing events (projects out unpredictable)")
print("  - 'zero' strategy simpler but can't handle unpredictable variance")